In [1]:
import json
import re

def analyze_log_file(file_path, expected_schema=None, report_type=None, file_type=None):
    total_rows = 0
    skipped_rows = 0
    max_columns = 0
    max_row_data = None
    max_row_line = 0
    distinct_columns = set()

    # ===== REPORT TYPE EXTRA COLUMNS =====
    AUDIT_EXTRA = {
        "s3_path", "Provider", "Server", "Agent", "Service",
        "TransactTimeTs", "TradeDateTs", "IngestedForDate", "OrderDate",
        "CompleteQty", "CompleteAvgPx", "CompleteLastShares", "CompleteCumQty",
        "CompleteLeaveQty", "CompleteLastPx"
    }

    ORDERS_EXTRA = {
        "s3_path", "Provider", "Server", "Agent", "Service",
        "TransactTimeTs", "IngestedForDate", "OrderDate",
        "CompleteLeaveQty", "PercentExecuted", "CompleteQty",
        "CompleteAvgPx", "CompleteCumQty"
    }

    EXC_EXTRA = {
        "s3_path", "Provider", "Server", "Agent", "Service",
        "TransactTimeTs", "TradeDateTs", "IngestedForDate", "OrderDate",
        "CompleteQty", "CompleteLeaveQty", "CompleteCumQty",
        "CompleteLastShares", "CompleteAvgPx", "PercentExecuted"
    }

    REPORT_MAP = {
        "AUDIT": AUDIT_EXTRA,
        "ORDERS": ORDERS_EXTRA,
        "EXC": EXC_EXTRA
    }

    # ===== REPORT TYPE → Cache filter mapping =====
    CACHE_FILTER_MAP = {
        "ORDERS": {"Orders"},
        "EXC":    {"Executions"},
        "AUDIT":  None   # auditdas me Cache filter nahi
    }

    file_type_upper  = file_type.strip().lower()  if file_type   else None
    report_type_upper = report_type.strip().upper() if report_type else None

    # Cache filter decide karo
    cache_filter = None
    if file_type_upper == "updatedas" and report_type_upper in CACHE_FILTER_MAP:
        cache_filter = CACHE_FILTER_MAP[report_type_upper]

    def clean_json(json_str):
        return re.sub(r'[\x00-\x1F\x7F]', '', json_str)

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            try:
                parts = line.split('|', 2)
                if len(parts) < 3:
                    continue
                json_part = parts[2].strip()
                if json_part.endswith('|'):
                    json_part = json_part[:-1].strip()
                json_part = clean_json(json_part)
                data = json.loads(json_part)

                # ===== FILTER: updatedas + report_type ke hisaab se Cache check =====
                if cache_filter is not None:
                    if data.get("Cache", "") not in cache_filter:
                        skipped_rows += 1
                        continue

                total_rows += 1
                col_count = len(data)
                if col_count > max_columns:
                    max_columns = col_count
                    max_row_data = data
                    max_row_line = line_no
                distinct_columns.update(data.keys())

            except Exception as e:
                print(f"⚠️ Skipping line {line_no}: {e}")

    if max_row_data is None:
        print("❌ Koi bhi valid row nahi mili matching filter ke saath!")
        return set()

    # ===== MERGE REPORT TYPE COLUMNS IF SPECIFIED =====
    extra_cols = REPORT_MAP.get(report_type_upper, set()) if report_type_upper else set()
    combined_distinct = distinct_columns | extra_cols

    # ===== OUTPUT =====
    print("\n================ FINAL RESULT ================\n")
    print(f"📁 File Type      : {file_type_upper if file_type_upper else 'Not specified'}")
    print(f"📄 Report Type    : {report_type_upper if report_type_upper else 'Not specified'}")
    if cache_filter is not None:
        print(f"🔍 Cache Filter   : {cache_filter}")
        print(f"⏭️  Skipped Rows   : {skipped_rows}")
    else:
        print(f"🔍 Cache Filter   : None (auditdas — no filter applied)")
    print(f"📊 Rows Processed : {total_rows}")
    print(f"\n🔥 Max Columns Found : {max_columns}")
    print(f"📍 Found at Line     : {max_row_line}")
    print("\n📋 Columns in Max Row:\n")
    for i, (key, value) in enumerate(max_row_data.items(), start=1):
        print(f"  {i}. {key} : {value}")

    print(f"\n🧠 Total Distinct Columns (from log): {len(distinct_columns)}")

    if report_type_upper and extra_cols:
        added = extra_cols - distinct_columns
        print(f"\n➕ Extra Columns Added from {report_type_upper}: {len(added)}")
        if added:
            for col in sorted(added):
                print(f"   + {col}")
        print(f"\n🔢 Combined Distinct Columns (log + {report_type_upper}): {len(combined_distinct)}")
    else:
        print("ℹ️  No report_type specified — skipping extra column merge.")

    print("\n📌 Combined Distinct Column Names:\n")
    for i, col in enumerate(sorted(combined_distinct), start=1):
        print(f"  {i}. {col}")

    # ===== ICEBERG VALIDATION =====
    if expected_schema:
        expected_set = set(expected_schema)
        missing_in_iceberg = combined_distinct - expected_set
        extra_in_iceberg = expected_set - combined_distinct

        print("\n================ SCHEMA VALIDATION ================\n")
        print(f"❌ Missing in Iceberg ({len(missing_in_iceberg)}):")
        for col in sorted(missing_in_iceberg):
            print(f"  - {col}")
        print(f"\n⚠️  Extra in Iceberg ({len(extra_in_iceberg)}):")
        for col in sorted(extra_in_iceberg):
            print(f"  - {col}")

    return combined_distinct


# ====================================================
# 🔹 YOUR FILE PATH
file_path = "/home/shariq/Downloads/Request-audittrail-22-04-2026.log"

# 🔹 FILE TYPE: "auditdas" | "updatedas"
file_type = "auditdas"

# 🔹 REPORT TYPE: "AUDIT" | "ORDERS" | "EXC" | None
report_type = "AUDIT"
#   ORDERS  →  sirf Cache="Orders"      wali rows
#   EXC     →  sirf Cache="Executions"  wali rows
#   AUDIT   →  koi Cache filter nahi (auditdas file)

# 🔹 OPTIONAL: Iceberg schema
expected_schema = [
    "MessageSequence", "MessageTag", "SubMessageTag", "ClOrdID",
    "OrderID", "OrderQty", "Price", "Side", "SymbolWithoutSfx",
    "TransactTime", "ExecType", "ExecID", "LastPx", "CumQty"
]

script_distinct_columns = analyze_log_file(file_path, expected_schema, report_type, file_type)


================ FINAL RESULT ================

📁 File Type      : auditdas
📄 Report Type    : AUDIT
🔍 Cache Filter   : None (auditdas — no filter applied)
📊 Rows Processed : 439548

🔥 Max Columns Found : 83
📍 Found at Line     : 148265

📋 Columns in Max Row:

  1. MessageSequence : 148265
  2. MessageTag : 8
  3. SubMessageTag : 0
  4. HdrPossResend : False
  5. PossDuplicate : False
  6. Pad : 0
  7. MessageLength : 715
  8. NumberOfFields : 65
  9. HdrOrderID : 8354102
  10. HdrBoothID : ['I', 'E', 'B', '\x00']
  11. VersionMaj : 9
  12. VersionMin : 0
  13. ABCD : 0
  14. INET_Address : ['0', '.', '0', '.', '0', '.', '0', '\x00', '\x02', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00']
  15. HdrSymbol : ['A', 'S', 'M', 'L', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00']
  16. MarketLookID : ['I', 'E', 'B', 'R', 'E', 'T', 'A', 'I', 'L', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', '\x00', 'B']
  17. ByteOrder : 
  18. Pad2 :  
  19. ClOrdID : IEB

In [3]:
import pandas as pd

def compare_with_csv(script_columns: set, csv_path: str, field_col: str = "field_name"):
    """
    Compares script's distinct columns with CSV's field_name column.
    """

    # Load CSV
    df = pd.read_csv(csv_path)

    # Normalize column headers (strip spaces, lowercase for matching)
    df.columns = df.columns.str.strip()

    # Find the field_name column (case-insensitive fallback)
    actual_col = None
    for col in df.columns:
        if col.lower() == field_col.lower():
            actual_col = col
            break

    if actual_col is None:
        print(f"❌ Column '{field_col}' not found in CSV!")
        print(f"   Available columns: {list(df.columns)}")
        return

    # Get CSV values — strip whitespace, drop nulls
    csv_columns = set(df[actual_col].dropna().astype(str).str.strip())

    # Normalize script columns too (strip whitespace just in case)
    script_cols_clean = set(c.strip() for c in script_columns)

    # Compute differences
    extra_in_script = script_cols_clean - csv_columns   # script has it, CSV doesn't
    extra_in_csv    = csv_columns - script_cols_clean   # CSV has it, script doesn't
    matched         = script_cols_clean & csv_columns   # both have it

    # ===== OUTPUT =====
    print("\n================ CSV COMPARISON RESULT ================\n")
    print(f"📜 Script Distinct Columns : {len(script_cols_clean)}")
    print(f"📄 CSV '{actual_col}' Columns : {len(csv_columns)}")
    print(f"✅ Matched                 : {len(matched)}")

    print(f"\n🟡 Script EXTRA ({len(extra_in_script)})  — in script but MISSING from CSV:")
    if extra_in_script:
        for i, col in enumerate(sorted(extra_in_script), 1):
            print(f"  {i}. Script  →  {col}")
    else:
        print("  ✅ None — all script columns exist in CSV")

    print(f"\n🔵 CSV EXTRA ({len(extra_in_csv)})  — in CSV but MISSING from script:")
    if extra_in_csv:
        for i, col in enumerate(sorted(extra_in_csv), 1):
            print(f"  {i}. CSV  →  {col}")
    else:
        print("  ✅ None — all CSV columns exist in script output")

    # ===== MATCHED LIST (optional, uncomment if needed) =====
    # print(f"\n✅ Matched Columns ({len(matched)}):")
    # for i, col in enumerate(sorted(matched), 1):
    #     print(f"  {i}. {col}")


# ====================================================
# 🔹 PATH TO YOUR CSV FILE
csv_path = "/home/shariq/Downloads/data_configurator_puite_audittrail_schema.csv"   # <-- apna path dalo

# 🔹 Run comparison using columns from Cell 1
compare_with_csv(script_distinct_columns, csv_path)


================ CSV COMPARISON RESULT ================

📜 Script Distinct Columns : 119
📄 CSV 'field_name' Columns : 144
✅ Matched                 : 119

🟡 Script EXTRA (0)  — in script but MISSING from CSV:
  ✅ None — all script columns exist in CSV

🔵 CSV EXTRA (25)  — in CSV but MISSING from script:
  1. CSV  →  ComplexOrderFields
  2. CSV  →  ComplianceID
  3. CSV  →  Contactname
  4. CSV  →  ContraTrader
  5. CSV  →  CoveredOrUncovered
  6. CSV  →  CoveredOrUncoveredDesc
  7. CSV  →  DeliverToCompID
  8. CSV  →  ExecutionCommission
  9. CSV  →  ExecutionTradeFee
  10. CSV  →  IDSource
  11. CSV  →  IsComplexTrade
  12. CSV  →  LegRatio
  13. CSV  →  LegRefID
  14. CSV  →  LocateReqd
  15. CSV  →  Locationid
  16. CSV  →  MaxFloor
  17. CSV  →  MinQty
  18. CSV  →  OnBehalfOfSubID
  19. CSV  →  OrderCommission
  20. CSV  →  OrderTradeFee
  21. CSV  →  OrigSendingTime
  22. CSV  →  PossDupFlag
  23. CSV  →  SecurityID
  24. CSV  →  StatusDesc
  25. CSV  →  WorkableQty
